# Запуск MT-обучения в osiris: режим проверки и полноценный запуск

Архитектура: общая CR-голова + канальная поправка (адаптер), у CR нет канальных башен, CTR-башня остаётся.

**Порядок работы**
1. Ячейки 1–3 один раз за сессию (импорты, Kerberos, параметры).
2. `RUN_MODE = "check"` → preflight → `DRY_RUN = False` → запуск → автоотчёт по логам.
3. Если отчёт без замечаний: `RUN_MODE = "full"`, `CONFIRM_FULL = True`, запуск.

Режим `check`: 1% файлов, 1 эпоха, 1 нода × 4 GPU, частое логирование и градиентный конфликт. Режим `full`: все файлы, 2 эпохи, 3 ноды × 4 GPU. Пресеты правятся в словаре `PRESETS`.

In [1]:
import os
import sys
import re
import glob
import json
import time
import shutil
import getpass
import subprocess

sys.path.insert(0, os.path.abspath("/opt/clients"))
import osiris

## 2. Kerberos / Hadoop (один раз за сессию)

In [2]:
NFS = "/home/datalab/nfs"
PREPARE_ENV = True   # False, если ticket и конфиги уже подготовлены

if PREPARE_ENV:
    for name in ("hadoop", "nss", "krb5cc", "krb5.conf"):
        path = f"{NFS}/{name}"
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)
    shutil.copytree("/etc/hadoop/conf", f"{NFS}/hadoop", dirs_exist_ok=True)
    shutil.copy("/etc/krb5.conf", f"{NFS}/krb5.conf")
    subprocess.run(["kinit"], input=getpass.getpass("Пароль для kinit: ") + "\n", text=True, check=True)
    ticket = max(glob.glob("/tmp/krb5cc_*"), key=os.path.getmtime)
    shutil.copy(ticket, f"{NFS}/krb5cc")
    os.makedirs(f"{NFS}/nss", exist_ok=True)
    shutil.copy("/etc/passwd", f"{NFS}/nss/")
    print("Kerberos/Hadoop подготовлены, ticket:", ticket)

Пароль для kinit:  ········


Password for 21835579_omega-sbrf-ru@DF.SBRF.RU: 
Kerberos/Hadoop подготовлены, ticket: /tmp/krb5cc_1000290000


## 3. Параметры

In [8]:
# ============ РЕЖИМ ============
RUN_MODE = "full"        # "check" — проверка; "full" — полноценное обучение
USE_MULTITASK = True      # True: MT-схема; False: прежний запуск --joint (для сравнения)
DRY_RUN = True            # True: только печать аргументов, задача не создаётся
CONFIRM_FULL = True      # защита от случайного полного запуска

PRESETS = {
    "check": dict(file_fraction=0.01, epochs=1, num_nodes=1, num_gpus=4, log_every=10, gc_every=50, suffix="_check"),
    "full":  dict(file_fraction=1.0,  epochs=2, num_nodes=4, num_gpus=4, log_every=50, gc_every=0,  suffix=""),
}

# ============ ПУТИ ============
SCRIPT_PATH = "/home/datalab/nfs/Omni/TRAIN OMNI/5 MDMH/Train CR CTR EXPERTS CHNL/mdmh_cr_ctr_base_train.py"
OMNI_MAIN = "/home/datalab/nfs/Omni/omnichannel_main"          # PATH_OMNI_MAIN из .env
LOCAL_ROOT = "/home/datalab/nfs/Omni/"
CONFIG_PATH = "/home/datalab/nfs/Omni/configs_weights/omni_config.json"
DATASET_PATH = ""
MDL_BASE = "omni_val_cr_ctr_mt_adapter"

# ============ ДАННЫЕ ============
TRAIN_TABLE = "sdi_cr_ctr_dataset_val"
VAL_TABLE = ""
TST_TABLE = ""
OOT_TABLE = ""
DATA_FORMAT = "raw"
ROWS_CNT = 49_637_381          # строк в train
BATCH_SIZE = 128               # на один процесс
SEED = 42
UNK_EMB_PROB = 0.1
NUM_WORKERS = 2
TARGET_COLUMNS = "trgt_fin,trgt_ctr"
HEAD_TARGETS = '{"cr": "trgt_fin", "ctr": "trgt_ctr"}'
HEAD_WEIGHTS = '{"cr": 1.0, "ctr": 1.0}'

# ============ MT-АРХИТЕКТУРА ============
CHANNELS_PER_TASK = '{"cr": 4, "ctr": 1}'
EVALUATE_COLS = "eval_prod,eval_channel"
CHANNEL_NAMES = '{"cr": ["mass", "phygital", "premier", "sbol"]}'      # порядок = channel_ids
MT_CONFIG = {
    "channel_ids": {"cr": [0, 1, 2, 3]},                               # mass=0, phygital=1, premier=2, sbol=3
    "n_task_experts": {"cr": 2, "ctr": 2}, "n_shared_experts": 2,
    "expert_hidden": 512, "expert_dim": 256, "tower_hidden": 128,
    "tower_type": "none",                                               # у CR нет канальных башен, башня CTR остаётся
    "general_tasks": ["cr"],
    "adapter_tasks": ["cr"], "adapter_emb_dim": 8, "adapter_hidden": 32,
    "adapter_grad": "isolated",                                         # поправка не влияет на базу
    "adapter_loss_weight": 1.0, "adapter_reg": 0.01,
}
MT_LRS = '{"experts": 1e-3, "gates": 5e-4, "towers": 1e-3, "adapter": 1e-3}'
MT_WDS = '{"experts": 1e-2, "gates": 1e-3, "towers": 1e-2, "adapter": 1e-2}'
NO_FIND_UNUSED = False

# ============ ПРЕЖНИЙ РЕЖИМ (USE_MULTITASK=False) ============
HEAD_KINDS = "cr,ctr"
CHANNEL_SPLIT = True

# ============ ЭМБЕДДИНГИ / ПОДМОДЕЛИ ============
EVK_WEIGHTS_PATH = "evk_weights.pt"
DLT_WEIGHTS_PATH = "dlt_weights.pt"
USE_EMB_FLAGS = """{"romashka_emb": true,
        "evk_emb": true,
        "prod_hier_emb": true,
        "prod_desc_emb": true
    }"""

# ============ RESUME / ЗАМОРОЗКА ============
RESUME = None                  # путь к .pt
NO_LOAD_OPTIMIZER = False      # True, если у чекпоинта другое число param-групп
START_EPOCH = 0
ADAPTER_ONLY = False           # True: учить только mt.adapters (базу грузить через RESUME)
FREEZE_BACKBONE = False

# ============ РЕСУРСЫ ============
POOL = "b2c"

# ============ ПРОИЗВОДНЫЕ ============
assert RUN_MODE in PRESETS, f"RUN_MODE должен быть одним из {list(PRESETS)}"
P = PRESETS[RUN_MODE]
FILE_FRACTION, EPOCHS = P["file_fraction"], P["epochs"]
NUM_NODES, NUM_GPUS = P["num_nodes"], P["num_gpus"]
NUM_PROCESS = NUM_NODES * NUM_GPUS
MDL_NAME = MDL_BASE + P["suffix"]
ROWS_CNT_EFF = max(1, int(ROWS_CNT * FILE_FRACTION))      # расписание LR считается от реального числа строк
EXPECTED_TOTAL_PARAMS = 113_405_383                       # расчёт для D=6168 и конфигурации выше; для другой конфигурации игнорируйте
print(f"mode={RUN_MODE} multitask={USE_MULTITASK} model={MDL_NAME} nodes={NUM_NODES} procs={NUM_PROCESS} "
      f"rows_cnt={ROWS_CNT_EFF} epochs={EPOCHS} file_fraction={FILE_FRACTION}")

mode=full multitask=True model=omni_val_cr_ctr_mt_adapter nodes=4 procs=16 rows_cnt=49637381 epochs=2 file_fraction=1.0


## 4. Preflight

Проверяет JSON-параметры, согласованность задач и каналов, валидирует `MT_CONFIG` через `MultiTaskConfig` (если доступен импорт), проверяет, что в `SCRIPT_PATH` и в `OMNI_MAIN` применены нужные правки (в том числе передача `mt_config` в модель, из-за пропуска которой падал первый запуск).

In [9]:
def _fail(msg):
    raise AssertionError("PREFLIGHT: " + msg)

def preflight():
    parsed = {}
    for name in ("HEAD_TARGETS", "HEAD_WEIGHTS", "CHANNELS_PER_TASK", "CHANNEL_NAMES", "MT_LRS", "MT_WDS", "USE_EMB_FLAGS"):
        try:
            parsed[name] = json.loads(globals()[name])
        except Exception as e:
            _fail(f"{name} не парсится как JSON: {e}")
    cols = [c.strip() for c in TARGET_COLUMNS.split(",")]
    for task, col in parsed["HEAD_TARGETS"].items():
        if col not in cols:
            _fail(f"HEAD_TARGETS[{task}]={col} нет в TARGET_COLUMNS={cols}")
    if set(parsed["HEAD_WEIGHTS"]) != set(parsed["HEAD_TARGETS"]):
        _fail("ключи HEAD_WEIGHTS и HEAD_TARGETS расходятся")

    if not os.path.exists(SCRIPT_PATH):
        _fail(f"нет SCRIPT_PATH={SCRIPT_PATH}")
    text = open(SCRIPT_PATH, encoding="utf-8").read()

    if USE_MULTITASK:
        cps = parsed["CHANNELS_PER_TASK"]
        if set(cps) != set(parsed["HEAD_TARGETS"]):
            _fail("задачи CHANNELS_PER_TASK и HEAD_TARGETS расходятся")
        for task, k in cps.items():
            names = parsed["CHANNEL_NAMES"].get(task)
            if names is not None and len(names) != k:
                _fail(f"CHANNEL_NAMES[{task}] должен содержать {k} имён")
        bad = set(parsed["MT_LRS"]) - {"experts", "gates", "towers", "adapter"}
        bad |= set(parsed["MT_WDS"]) - {"experts", "gates", "towers", "adapter"}
        if bad:
            _fail(f"MT_LRS/MT_WDS: неизвестные группы {bad}")
        if len([c for c in EVALUATE_COLS.split(",") if c.strip()]) < 2:
            _fail("EVALUATE_COLS должен содержать минимум 2 колонки: [prod, channel]")
        for tok in ("--multitask", "--channels-per-task", "--evaluate-cols", "--mt-lrs", "cycles_mt"):
            if tok not in text:
                _fail(f"в SCRIPT_PATH нет '{tok}': применены ли патчи к скрипту?")
        if not re.search(r"mt_config\s*=\s*mt_config", text):
            _fail("в SCRIPT_PATH модель создаётся без mt_config=mt_config (OmnichannelModel(..., mt_config=mt_config))")
        if MT_CONFIG.get("adapter_tasks") and "--train-adapter-only" not in text:
            print("WARN: в SCRIPT_PATH нет --train-adapter-only (stage5.patch не применён к скрипту)")
        if '(_ck.get("extra_data") or {})' not in text:
            print("WARN: resume не восстанавливает scheduler (правка F6 не применена)")

        src = f"{OMNI_MAIN}/src/omnichannel_model/omni"
        uf = f"{src}/utils.py"
        if MT_CONFIG.get("adapter_tasks") and os.path.exists(uf) and "mt.adapters." not in open(uf, encoding="utf-8").read():
            print("WARN: в utils.py нет группы mt.adapters: параметры адаптера попадут в группу experts")

        for f, tok in (("multitask.py", "ChannelAdapter" if MT_CONFIG.get("adapter_tasks") else "class MultiTaskHead"),
                       ("cycles_mt.py", "adapter_ids" if MT_CONFIG.get("adapter_tasks") else "def train_multitask_osiris")):
            fp = f"{src}/{f}"
            if not os.path.exists(fp):
                print(f"WARN: нет {fp} (проверьте OMNI_MAIN)")
            elif tok not in open(fp, encoding="utf-8").read():
                _fail(f"{fp} не содержит '{tok}': файл не обновлён")
        cfg_dict = dict(MT_CONFIG)
        cfg_dict.update(tasks=list(cps), channels_per_task=cps,
                        target_cols={h: cols.index(c) for h, c in parsed["HEAD_TARGETS"].items()})
        try:
            sys.path.insert(0, f"{OMNI_MAIN}/src")
            from omnichannel_model.omni.multitask import MultiTaskConfig
            MultiTaskConfig.from_dict(cfg_dict)
            print("MT_CONFIG валиден (MultiTaskConfig)")
        except ImportError as e:
            print("WARN: MultiTaskConfig не импортируется в ноутбуке, валидация конфига пропущена:", e)
        except Exception as e:
            _fail(f"MT_CONFIG не проходит валидацию: {e}")
    if RESUME and not os.path.exists(RESUME):
        _fail(f"нет RESUME={RESUME}")
    if ADAPTER_ONLY and not RESUME:
        _fail("ADAPTER_ONLY=True без RESUME: у базы не будет обученных весов")
    print("PREFLIGHT: OK")
    return parsed

PARSED = preflight()

MT_CONFIG валиден (MultiTaskConfig)
PREFLIGHT: OK


## 5. Аргументы и задача osiris

In [10]:
def build_script_args():
    a = [
        "--num_machines", f"{NUM_NODES}",
        "--num_processes", f"{NUM_PROCESS}",
        "--main_process_ip", "$(MASTER_ADDR).datalab.svc.cluster.local",
        "--main_process_port", "$(MASTER_PORT)",
        "--machine_rank", "$(RANK)",
        f"{SCRIPT_PATH}",
        "--local-root", str(LOCAL_ROOT),
        "--config-path", str(CONFIG_PATH),
        "--mdl-name", str(MDL_NAME),
        "--data-format", str(DATA_FORMAT),
        "--train-table", str(TRAIN_TABLE),
        "--rows-cnt", str(ROWS_CNT_EFF),
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--seed", str(SEED),
        "--num-workers", str(NUM_WORKERS),
        "--file-fraction", str(FILE_FRACTION),
        "--unk-emb-prob", str(UNK_EMB_PROB),
        "--target-columns", str(TARGET_COLUMNS),
        "--head-targets", HEAD_TARGETS,
        "--head-weights", HEAD_WEIGHTS,
        "--evk-weights-path", EVK_WEIGHTS_PATH,
        "--dlt-weights-path", DLT_WEIGHTS_PATH,
        "--use-emb-flags", str(USE_EMB_FLAGS),
        "--start-epoch", str(START_EPOCH),
    ]
    if DATASET_PATH:
        a += ["--dataset-path", DATASET_PATH]
    for flag, val in (("--val-table", VAL_TABLE), ("--tst-table", TST_TABLE), ("--oot-table", OOT_TABLE)):
        if val:
            a += [flag, val]
    if USE_MULTITASK:
        a += ["--multitask",
              "--channels-per-task", CHANNELS_PER_TASK,
              "--mt-config", json.dumps(MT_CONFIG),
              "--evaluate-cols", EVALUATE_COLS,
              "--channel-names", CHANNEL_NAMES,
              "--mt-lrs", MT_LRS, "--mt-wds", MT_WDS,
              "--log-every", str(P["log_every"]),
              "--gc-every", str(P["gc_every"])]
        if NO_FIND_UNUSED:
            a += ["--no-find-unused"]
        if ADAPTER_ONLY:
            a += ["--train-adapter-only"]
    else:
        a += ["--head-kinds", str(HEAD_KINDS), "--joint", "--log-every", str(P["log_every"])]
        if CHANNEL_SPLIT:
            a += ["--channel-split"]
    if RESUME:
        a += ["--resume-from", str(RESUME)]
        if NO_LOAD_OPTIMIZER:
            a += ["--no-load-optimizer"]
    if FREEZE_BACKBONE:
        a += ["--freeze-backbone"]
    return a


def create_job(job_name, num_gpus):
    return osiris.create(
        name=f"sc-{job_name}",
        image="registry.ca.sbrf.ru/ci02684173/ci02697916/notebooks/python3.12/cuda12.4/d-03.000.00:d-03.000.00-gigachat",
        restart=False,
        pool=POOL,
        script=SCRIPT_PATH,
        command=["accelerate", "launch"],
        args=build_script_args(),
        envs={
            "PYTHONPATH": "",
            "OMP_NUM_THREADS": "30",
            "NCCL_DEBUG": "INFO",
            "HADOOP_CONF_DIR": "/home/datalab/nfs/hadoop",
            "KRB5_CONFIG": "/home/datalab/nfs/krb5.conf",
            "KRB5CCNAME": "/home/datalab/nfs/krb5cc",
            "ARROW_LIBHDFS_OPTS": "-Djava.security.krb5.conf=/home/datalab/nfs/krb5.conf",
            "LIBHDFS_OPTS": "-Djava.security.krb5.conf=/home/datalab/nfs/krb5.conf",
            "HADOOP_OPTS": "-Djava.security.krb5.conf=/home/datalab/nfs/krb5.conf",
            "LD_PRELOAD": "/usr/lib64/libnss_wrapper.so",
            "NSS_WRAPPER_PASSWD": "/home/datalab/nfs/nss/passwd",
            "NSS_WRAPPER_GROUP": "/etc/group",
            "CLASSPATH": f":/home/datalab/nfs/hadoop:" + os.environ["CLASSPATH"].split(":", 2)[-1],
            "TORCH_DISTRIBUTED_DEBUG": "DETAIL",
            "NCCL_SOCKET_FAMILY": "AF_INET",
            "GLOO_SOCKET_FAMILY": "AF_INET",
            "NCCL_SOCKET_IFNAME": "eth0",
            "GLOO_SOCKET_IFNAME": "eth0",
            "NCCL_IB_DISABLE": "1",
            "NCCL_P2P_DISABLE": "1",
            "NCCL_SHM_DISABLE": "1",
        },
        num_nodes=NUM_NODES,
        num_gpus=num_gpus,
        type="pytorchjob",
    )

print(" ".join(f"'{x}'" if (" " in x or x[:1] in "{[") else x for x in build_script_args()))

--num_machines 4 --num_processes 16 --main_process_ip $(MASTER_ADDR).datalab.svc.cluster.local --main_process_port $(MASTER_PORT) --machine_rank $(RANK) '/home/datalab/nfs/Omni/TRAIN OMNI/5 MDMH/Train CR CTR EXPERTS CHNL/mdmh_cr_ctr_base_train.py' --local-root /home/datalab/nfs/Omni/ --config-path /home/datalab/nfs/Omni/configs_weights/omni_config.json --mdl-name omni_val_cr_ctr_mt_adapter --data-format raw --train-table sdi_cr_ctr_dataset_val --rows-cnt 49637381 --epochs 2 --batch-size 128 --seed 42 --num-workers 2 --file-fraction 1.0 --unk-emb-prob 0.1 --target-columns trgt_fin,trgt_ctr --head-targets '{"cr": "trgt_fin", "ctr": "trgt_ctr"}' --head-weights '{"cr": 1.0, "ctr": 1.0}' --evk-weights-path evk_weights.pt --dlt-weights-path dlt_weights.pt --use-emb-flags '{"romashka_emb": true,
        "evk_emb": true,
        "prod_hier_emb": true,
        "prod_desc_emb": true
    }' --start-epoch 0 --multitask --channels-per-task '{"cr": 4, "ctr": 1}' --mt-config '{"channel_ids": {"cr": [

## 6. Запуск

`DRY_RUN = True` ничего не создаёт. Для `RUN_MODE = "full"` нужно ещё `CONFIRM_FULL = True`. В логе мастера сразу после старта ищите строки `[MODEL] Total params` и `[MT] первый батч`.

In [11]:
DRY_RUN = False

In [12]:
JOB_NAME = f"train-cr-ctr-mt-{RUN_MODE}" if USE_MULTITASK else f"train-cr-ctr-{RUN_MODE}"

if DRY_RUN:
    print("DRY_RUN: задача не создавалась")
else:
    if RUN_MODE == "full" and not CONFIRM_FULL:
        raise RuntimeError("Для полноценного запуска установите CONFIRM_FULL = True (после успешного режима check)")
    PARSED = preflight()
    job = create_job(JOB_NAME, NUM_GPUS)
    print(job)
    job_name = job["job"]

MT_CONFIG валиден (MultiTaskConfig)
PREFLIGHT: OK
{'job': 'training-21835579-pytorchjob-sc-train-cr-ctr-mt-full', 'uuid': '686081ec-b5ef-432c-be21-854c970e52ec', 'state': 'Created', 'time': '2026-10-18T17:02:54Z', 'projects': []}


## 7. Мониторинг и автоотчёт по логам

In [13]:
def _find_job(name):
    for j in osiris.list()["jobs"]:
        if j["job"] == name:
            return j
    return None


def check_report(text, expected_total=EXPECTED_TOTAL_PARAMS):
    """Автоматическая проверка логов мастера. Возвращает список замечаний (пусто = всё в порядке)."""
    issues = []
    m = re.search(r"\[MODEL\] Total params: ([\d,]+)\s+Trainable: ([\d,]+)", text)
    if not m:
        issues.append("нет строки [MODEL] Total params")
    else:
        total, trainable = (int(x.replace(",", "")) for x in m.groups())
        print(f"параметры: всего {total:,}, обучаемых {trainable:,}")
        if USE_MULTITASK and total != expected_total:
            issues.append(f"всего параметров {total:,}, ожидалось {expected_total:,} (если конфигурация менялась, это нормально)")
    fb = re.search(r"\[MT\] первый батч: (.*)", text)
    if USE_MULTITASK:
        if not fb:
            issues.append("нет строки [MT] первый батч")
        else:
            line = fb.group(1)
            print("первый батч:", line)
            mc = re.search(r"ctr: labeled=(\d+)/(\d+)", line)
            if mc and not (15 <= int(mc.group(1)) <= 70):
                issues.append(f"в первом батче {mc.group(1)} CTR-строк из {mc.group(2)}: ожидается около 38 при перемешанных данных")
            if "неизвестные каналы" in line:
                issues.append("в данных есть каналы вне channel_ids")
    if "Traceback" in text:
        issues.append(f"в логе есть Traceback ({text.count('Traceback')})")
    if "non-finite loss" in text:
        issues.append("встретился нечисловой лосс")
    if "Error processing file" in text:
        issues.append("ошибки чтения файлов (Error processing file): часть данных могла быть пропущена")
    if "[CKPT] Saved" not in text:
        issues.append("нет строки [CKPT] Saved: чекпоинт мог не сохраниться")
    last = None
    for last in re.finditer(r"Epoch \d+, Batch \d+, Loss: [^\r\n]*", text):
        pass
    if last:
        print("последняя строка прогресса:", last.group(0)[:200])
        if USE_MULTITASK:
            for key in ("ctr", "cr_general") + (("cr_adj",) if MT_CONFIG.get("adapter_tasks") else ()):
                if f"{key}:" not in last.group(0):
                    issues.append(f"в строке прогресса нет лосса '{key}'")
    else:
        issues.append("нет строк прогресса (обучение не стартовало?)")
    return issues


def metrics_commands():
    d = os.path.dirname(SCRIPT_PATH)
    wdir = f"{LOCAL_ROOT.rstrip('/')}/omnichannel_artifacts/omni/weights"
    chk = f"{MDL_NAME}_ep{EPOCHS - 1}_weights.pt"
    for head in ("cr_general", "cr_adj", "ctr") if MT_CONFIG.get("adapter_tasks") else ("cr", "ctr"):
        print(f"accelerate launch {d}/base_ddp_metrics.py --config-path {CONFIG_PATH} --mdl-name {MDL_NAME} "
              f"--chkpt-name {chk} --multitask --mt-config {wdir}/{MDL_NAME}_mt_config.json "
              f"--target-columns {TARGET_COLUMNS} --head-name {head} --eval-splits val --val-table <ТАБЛИЦА_VAL> "
              f"--use-emb-flags '{' '.join(USE_EMB_FLAGS.split())}'")


if not DRY_RUN:
    my_job = _find_job(job_name)
    while my_job is None or my_job["state"] not in ("Succeeded", "Failed"):
        my_job = _find_job(job_name)
        if my_job is not None:
            print(my_job["state"], end="\r")
        time.sleep(5)
    print("Состояние:", my_job["state"])
    try:
        out = osiris.logs(job_name, timestamps=False, is_master=True, tail_lines=20000)
        text = out if isinstance(out, str) else (str(out) if out is not None else "")
    except Exception as e:
        text = ""
        print("не удалось получить логи автоматически:", e)
    if text:
        issues = check_report(text)
        print("\nЗамечания:" if issues else "\nЗамечаний нет")
        for i in issues:
            print(" -", i)
    else:
        print("Логи не получены строкой: посмотрите их в следующей ячейке и проверьте вручную")
    if my_job["state"] == "Succeeded":
        print("\nКоманды для расчёта метрик (подставьте таблицу val):")
        metrics_commands()

Состояние: Succeeded
последняя строка прогресса: Epoch 2, Batch 4601, Loss: 0.6483, ctr: 0.1951, cr_general: 0.2272, cr_adj: 0.2252, GradNorm: 0.115, LR: 7.79e-05:  17%|█▋        | 4632/26660 [59:38<4:27:58,  1.37it/s]', 'Epoch 2, Batch 4601, Loss: 

Замечания:
 - нет строки [MODEL] Total params
 - нет строки [MT] первый батч

Команды для расчёта метрик (подставьте таблицу val):
accelerate launch /home/datalab/nfs/Omni/TRAIN OMNI/5 MDMH/Train CR CTR EXPERTS CHNL/base_ddp_metrics.py --config-path /home/datalab/nfs/Omni/configs_weights/omni_config.json --mdl-name omni_val_cr_ctr_mt_adapter --chkpt-name omni_val_cr_ctr_mt_adapter_ep1_weights.pt --multitask --mt-config /home/datalab/nfs/Omni/omnichannel_artifacts/omni/weights/omni_val_cr_ctr_mt_adapter_mt_config.json --target-columns trgt_fin,trgt_ctr --head-name cr_general --eval-splits val --val-table <ТАБЛИЦА_VAL> --use-emb-flags '{"romashka_emb": true, "evk_emb": true, "prod_hier_emb": true, "prod_desc_emb": true }'
accelerate launch /

In [13]:
# логи мастера вручную (ищите [MODEL], [MT], Warning, Traceback)
osiris.logs(job_name, timestamps=False, is_master=True, tail_lines=300)

{'job': 'training-21835579-pytorchjob-sc-train-cr-ctr-mt-check',
 'pods': [{'pod': 'training-21835579-pytorchjob-sc-train-cr-ctr-mt-check-master-0',
   'state': 'Succeeded',
   'logs': ['',
    'Epoch 1, Batch 681, Loss: 0.7609, ctr: 0.2067, cr_general: 0.2769, cr_adj: 0.2764, GradNorm: 0.252, LR: 4.53e-05:  64%|██████▍   | 683/1065 [06:44<03:53,  1.63it/s]',
    'Epoch 1, Batch 681, Loss: 0.7609, ctr: 0.2067, cr_general: 0.2769, cr_adj: 0.2764, GradNorm: 0.252, LR: 4.53e-05:  64%|██████▍   | 684/1065 [06:45<03:45,  1.69it/s]',
    'Epoch 1, Batch 681, Loss: 0.7609, ctr: 0.2067, cr_general: 0.2769, cr_adj: 0.2764, GradNorm: 0.252, LR: 4.53e-05:  64%|██████▍   | 685/1065 [06:46<03:39,  1.73it/s]',
    'Epoch 1, Batch 681, Loss: 0.7609, ctr: 0.2067, cr_general: 0.2769, cr_adj: 0.2764, GradNorm: 0.252, LR: 4.53e-05:  64%|██████▍   | 686/1065 [06:46<03:35,  1.76it/s]',
    'Epoch 1, Batch 681, Loss: 0.7609, ctr: 0.2067, cr_general: 0.2769, cr_adj: 0.2764, GradNorm: 0.252, LR: 4.53e-05:  65

In [ ]:
# удаление задачи
# osiris.delete(job_name)